# Assignment 2

In [18]:
import os
import time
from typing import Optional, Dict, Tuple, Any

import gymnasium as gym
from gymnasium import spaces
from gymnasium.wrappers import RecordVideo
import numpy as np
import pygame
import matplotlib.pyplot as plt

print(f"Gymnasium version: {gym.__version__}")
print(f"NumPy version:     {np.__version__}")
print(f"Pygame version:    {pygame.__version__}")

Gymnasium version: 1.4.0
NumPy version:     2.5.4
Pygame version:    2.6.1


In [19]:
class CustomGridMazeEnv(gym.Env):
    metadata = {
        "render_modes": ["human", "rgb_array"],
        "render_fps": 4,
    }

    def __init__(self, size: int = 5, render_mode: Optional[str] = None):
        super().__init__()
        self.size = size
        self.render_mode = render_mode
        self.cell_size = 100
        self.window_size = self.size * self.cell_size
        self.window = None
        self.clock = None

        # Internal coordinates:
        self._agent_location = np.array([-1, -1], dtype=np.int32)
        self._target_location = np.array([-1, -1], dtype=np.int32)
        self._obstacle_locations = np.empty((2, 2), dtype=np.int32)
        self._layout_initialized = False
        self.P: Dict[int, Dict[int, list]] = {}

        self.observation_space = spaces.Dict(
            {
                "agent": spaces.Box(0, size - 1, shape=(2,), dtype=np.int32),
                "target": spaces.Box(0, size - 1, shape=(2,), dtype=np.int32),
                "obstacles": spaces.Box(0, size - 1, shape=(2, 2), dtype=np.int32),
            }
        )

        
        self.action_space = spaces.Discrete(4)
        self._action_to_direction = {
            0: np.array([0, 1]),   
            1: np.array([-1, 0]),  
            2: np.array([0, -1]),  
            3: np.array([1, 0]),   
        }

    def pos_to_state(self, pos) -> int:
        """Maps 2d into 1d index s = row * size + col."""
        return int(pos[0] * self.size + pos[1])

    def state_to_pos(self, state: int) -> np.ndarray:
        """Maps 1d back to 2d"""
        return np.array(divmod(int(state), self.size), dtype=np.int32)

    def generate_maze(self):
        """Samples 1 target and 2 distinct obstacles, then builds the transition table P."""
        cells = self.np_random.choice(self.size * self.size, size=3, replace=False)
        self._target_location = self.state_to_pos(cells[0])
        self._obstacle_locations = np.array(
            [self.state_to_pos(cells[1]), self.state_to_pos(cells[2])], dtype=np.int32
        )
        self._layout_initialized = True
        self._build_transition_table()

    def _build_transition_table(self):
        """Constructs transition table self.P[s][a] = [(prob, next_s, reward, terminated)]."""
        self.P = {s: {a: [] for a in range(self.action_space.n)} for s in range(self.size * self.size)}
        target_state = self.pos_to_state(self._target_location)
        obstacle_states = {self.pos_to_state(obs) for obs in self._obstacle_locations}

        for s in range(self.size * self.size):
            pos = self.state_to_pos(s)
            for a in range(self.action_space.n):
                # Terminal absorbing states
                if s == target_state or s in obstacle_states:
                    self.P[s][a] = [(1.0, s, 0.0, True)]
                else:
                    next_pos = np.clip(pos + self._action_to_direction[a], 0, self.size - 1)
                    next_s = self.pos_to_state(next_pos)
                    if next_s == target_state:
                        self.P[s][a] = [(1.0, next_s, 1.0, True)]
                    elif next_s in obstacle_states:
                        self.P[s][a] = [(1.0, next_s, -1.0, True)]
                    else:
                        self.P[s][a] = [(1.0, next_s, 0.0, False)]

    def _get_obs(self) -> Dict[str, np.ndarray]:
        return {
            "agent": self._agent_location.copy(),
            "target": self._target_location.copy(),
            "obstacles": self._obstacle_locations.copy(),
        }

    def _get_info(self) -> Dict[str, Any]:
        return {
            "distance": float(np.linalg.norm(self._agent_location - self._target_location, ord=1))
        }

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        new_layout = options.get("new_layout", False) if options else False

        # Generate new layout only if uninitialized 
        if not self._layout_initialized or new_layout:
            self.generate_maze()

        occupied = {
            self.pos_to_state(self._target_location),
            self.pos_to_state(self._obstacle_locations[0]),
            self.pos_to_state(self._obstacle_locations[1]),
        }
        empty_cells = [c for c in range(self.size * self.size) if c not in occupied]
        start_cell = self.np_random.choice(empty_cells)
        self._agent_location = self.state_to_pos(start_cell)

        return self._get_obs(), self._get_info()

    def step(self, action: int):
        direction = self._action_to_direction[action]
        self._agent_location = np.clip(self._agent_location + direction, 0, self.size - 1)

        hits_obstacle = any(
            np.array_equal(self._agent_location, obs) for obs in self._obstacle_locations
        )
        if hits_obstacle:
            terminated = True
            reward = -1.0
        else:
            terminated = bool(np.array_equal(self._agent_location, self._target_location))
            reward = 1.0 if terminated else 0.0

        return self._get_obs(), reward, terminated, False, self._get_info()

    def render(self):
        if self.render_mode is None:
            return None

        if self.window is None and self.render_mode == "human":
            pygame.init()
            pygame.display.init()
            self.window = pygame.display.set_mode((self.window_size, self.window_size))
            pygame.display.set_caption("Grid Maze")

        if self.clock is None and self.render_mode == "human":
            self.clock = pygame.time.Clock()

        canvas = pygame.Surface((self.window_size, self.window_size))
        canvas.fill((255, 255, 255))

        # Draw Obstacles (Red squares)
        for obs in self._obstacle_locations:
            rect = pygame.Rect(
                obs[1] * self.cell_size,
                obs[0] * self.cell_size,
                self.cell_size,
                self.cell_size,
            )
            pygame.draw.rect(canvas, (220, 60, 60), rect)

        # Draw Target (Green square)
        target_rect = pygame.Rect(
            self._target_location[1] * self.cell_size,
            self._target_location[0] * self.cell_size,
            self.cell_size,
            self.cell_size,
        )
        pygame.draw.rect(canvas, (60, 200, 60), target_rect)

        # Draw Agent (Blue circle)
        agent_center = (
            int((self._agent_location[1] + 0.5) * self.cell_size),
            int((self._agent_location[0] + 0.5) * self.cell_size),
        )
        pygame.draw.circle(canvas, (50, 110, 240), agent_center, int(self.cell_size * 0.35))

        # Draw Grid Lines
        for i in range(self.size + 1):
            pygame.draw.line(canvas, (210, 210, 210), (0, i * self.cell_size), (self.window_size, i * self.cell_size), 2)
            pygame.draw.line(canvas, (210, 210, 210), (i * self.cell_size, 0), (i * self.cell_size, self.window_size), 2)

        if self.render_mode == "human":
            self.window.blit(canvas, (0, 0))
            pygame.event.pump()
            pygame.display.update()
            self.clock.tick(self.metadata["render_fps"])
        elif self.render_mode == "rgb_array":
            return np.transpose(np.array(pygame.surfarray.pixels3d(canvas)), axes=(1, 0, 2))

    def close(self):
        if self.window is not None:
            pygame.display.quit()
            pygame.quit()
            self.window = None
            self.clock = None

In [22]:
# this is just a demo to make sure that the env is working 

# initialize the env in human mode (print on the screen)
env = CustomGridMazeEnv(render_mode="human")
# random seed that changes the position of target and obstacle
obs, info = env.reset(seed=38)
# render on screen
env.render()

# random demo 
reward =0
while (reward !=1):
    max_steps = 250
    for step_idx in range(1, max_steps + 1):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                env.close()
                raise SystemExit("Window closed by user.")

        action = env.action_space.sample()

        obs, reward, terminated, truncated, info = env.step(action)

        env.render()
        if terminated or truncated:
            if reward > 0:
                print(">>> SUCCESS: Agent reached the target! <<<")
            else:
                print(">>> GAME OVER: Agent hit an obstacle! <<<")
            time.sleep(1.5) 
            break
env.close()


(python:10028): Gtk-WARNING **: 15:54:20.879: gtk_disable_setlocale() must be called before gtk_init()


>>> SUCCESS: Agent reached the target! <<<
